# E-commerce Exploratory Data Analysis

## Objective

This notebook performs exploratory data analysis on the cleaned E-commerce Customer & Product Analytics dataset.

The analysis is designed from a **Business Analyst perspective**: identify revenue drivers, customer/product patterns, operational issues, profitability risks and actionable business insights.

### Business questions

1. How is revenue distributed across categories and regions?
2. Which categories and products contribute most to revenue and profit?
3. What customer segments generate the most value?
4. How do discounts affect revenue and profitability?
5. What is the return rate and where is it highest?
6. Which regions have delivery-time issues?
7. Which payment methods contribute most to sales?
8. Where are the major profitability risks?
9. What actions should management consider?


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Load the cleaned dataset

In [ ]:
# The notebook is stored in 03_Python/
# Therefore the cleaned data is expected at:
# ../01_Data/cleaned/ecommerce_cleaned.csv

file_path = "../01_Data/cleaned/ecommerce_cleaned.csv"

df = pd.read_csv(file_path)

df.head()


In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print("\nColumns:")
print(df.columns.tolist())


## 2. Data Quality Validation

In [ ]:
# Missing values
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0] if (missing > 0).any() else "No missing values found."


In [ ]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Date conversion
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

print("Invalid dates:", df["order_date"].isna().sum())
print("Date range:", df["order_date"].min().date(), "to", df["order_date"].max().date())


## 3. Descriptive Statistics

In [ ]:
df.describe(include="all").T


## 4. Derived Analytical Fields

In [ ]:
# Create analysis-friendly fields without changing the source dataset
df["return_flag"] = df["returned"].eq("Yes").astype(int)

df["age_group"] = pd.cut(
    df["customer_age"],
    bins=[17, 25, 35, 45, 55, 65, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66+"]
)

df["discount_pct"] = df["discount"] * 100

df["month"] = df["order_date"].dt.to_period("M").astype(str)
df["year"] = df["order_date"].dt.year

df[["order_date", "customer_age", "age_group", "discount", "discount_pct"]].head()


## 5. Revenue and Order Performance

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total Orders",
        "Unique Customers",
        "Unique Products",
        "Total Revenue",
        "Total Profit Metric",
        "Average Order Value",
        "Return Rate"
    ],
    "Value": [
        df["order_id"].nunique(),
        df["customer_id"].nunique(),
        df["product_id"].nunique(),
        df["total_amount"].sum(),
        df["profit_margin"].sum(),
        df["total_amount"].mean(),
        df["return_flag"].mean()
    ]
})

summary


In [ ]:
# Monthly revenue trend
monthly_revenue = (
    df.groupby("month", as_index=False)["total_amount"]
      .sum()
)

plt.figure(figsize=(14, 5))
plt.plot(monthly_revenue["month"], monthly_revenue["total_amount"], marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


## 6. Category Analysis

In [ ]:
category_summary = (
    df.groupby("category")
      .agg(
          orders=("order_id", "nunique"),
          revenue=("total_amount", "sum"),
          profit=("profit_margin", "sum"),
          avg_order_value=("total_amount", "mean"),
          return_rate=("return_flag", "mean")
      )
      .sort_values("revenue", ascending=False)
)

category_summary


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(category_summary.index, category_summary["revenue"])
plt.title("Revenue by Product Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 7. Customer Analysis

In [ ]:
customer_summary = (
    df.groupby("customer_id")
      .agg(
          orders=("order_id", "nunique"),
          revenue=("total_amount", "sum"),
          profit=("profit_margin", "sum"),
          avg_order_value=("total_amount", "mean"),
          return_rate=("return_flag", "mean")
      )
      .sort_values("revenue", ascending=False)
)

customer_summary.head(20)


In [ ]:
age_summary = (
    df.groupby("age_group", observed=False)
      .agg(
          orders=("order_id", "nunique"),
          revenue=("total_amount", "sum"),
          profit=("profit_margin", "sum"),
          return_rate=("return_flag", "mean")
      )
)

age_summary


## 8. Regional Analysis

In [ ]:
region_summary = (
    df.groupby("region")
      .agg(
          orders=("order_id", "nunique"),
          revenue=("total_amount", "sum"),
          profit=("profit_margin", "sum"),
          avg_delivery_days=("delivery_time_days", "mean"),
          return_rate=("return_flag", "mean")
      )
      .sort_values("revenue", ascending=False)
)

region_summary


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(region_summary.index, region_summary["revenue"])
plt.title("Revenue by Region")
plt.xlabel("Region")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()


## 9. Payment Method Analysis

In [ ]:
payment_summary = (
    df.groupby("payment_method")
      .agg(
          orders=("order_id", "nunique"),
          revenue=("total_amount", "sum"),
          profit=("profit_margin", "sum"),
          avg_order_value=("total_amount", "mean")
      )
      .sort_values("revenue", ascending=False)
)

payment_summary


## 10. Discount Analysis

In [ ]:
discount_summary = (
    df.groupby("discount")
      .agg(
          orders=("order_id", "nunique"),
          revenue=("total_amount", "sum"),
          profit=("profit_margin", "sum"),
          avg_order_value=("total_amount", "mean"),
          return_rate=("return_flag", "mean")
      )
      .sort_index()
)

discount_summary


In [ ]:
# Compare profitability across discount levels
plt.figure(figsize=(9, 5))
plt.plot(
    discount_summary.index * 100,
    discount_summary["profit"],
    marker="o"
)
plt.title("Profit Metric by Discount Level")
plt.xlabel("Discount (%)")
plt.ylabel("Total Profit Metric")
plt.tight_layout()
plt.show()


## 11. Return Analysis

In [ ]:
return_summary = (
    df.groupby("returned")
      .agg(
          orders=("order_id", "nunique"),
          revenue=("total_amount", "sum"),
          profit=("profit_margin", "sum")
      )
)

return_summary["order_share"] = (
    return_summary["orders"] / return_summary["orders"].sum()
)

return_summary


In [ ]:
return_by_category = (
    df.groupby("category")
      .agg(
          orders=("order_id", "nunique"),
          returned_orders=("return_flag", "sum"),
          return_rate=("return_flag", "mean")
      )
      .sort_values("return_rate", ascending=False)
)

return_by_category


## 12. Operational / Delivery Analysis

In [ ]:
delivery_summary = (
    df.groupby("region")
      .agg(
          avg_delivery_days=("delivery_time_days", "mean"),
          max_delivery_days=("delivery_time_days", "max"),
          orders=("order_id", "nunique"),
          return_rate=("return_flag", "mean")
      )
      .sort_values("avg_delivery_days", ascending=False)
)

delivery_summary


In [ ]:
# Delivery time distribution
plt.figure(figsize=(9, 5))
plt.hist(df["delivery_time_days"], bins=range(
    int(df["delivery_time_days"].min()),
    int(df["delivery_time_days"].max()) + 2
))
plt.title("Distribution of Delivery Time")
plt.xlabel("Delivery Time (days)")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()


## 13. Profitability Risk Analysis

In [ ]:
# Note: the source column is named profit_margin.
# We retain the original name for traceability.

profitability = (
    df.groupby("category")
      .agg(
          revenue=("total_amount", "sum"),
          profit_metric=("profit_margin", "sum"),
          orders=("order_id", "nunique"),
          avg_profit_metric=("profit_margin", "mean")
      )
)

profitability["profit_rate_vs_revenue"] = (
    profitability["profit_metric"] / profitability["revenue"]
)

profitability.sort_values("profit_metric")


In [ ]:
negative_profit = df[df["profit_margin"] < 0]

print("Orders with negative profit metric:", len(negative_profit))
print("Share of orders with negative profit metric:",
      f"{len(negative_profit) / len(df):.2%}")

negative_by_category = (
    negative_profit.groupby("category")
    .size()
    .sort_values(ascending=False)
    .rename("negative_profit_orders")
)

negative_by_category


## 14. Correlation Analysis

In [ ]:
numeric_cols = [
    "price",
    "discount",
    "quantity",
    "delivery_time_days",
    "total_amount",
    "shipping_cost",
    "profit_margin",
    "customer_age",
    "calculated_amount",
    "expected_amount",
    "amount_difference"
]

corr = df[numeric_cols].corr()

corr["total_amount"].sort_values(ascending=False)


## 15. Key Business Findings

In [ ]:
# Automatically generate a compact set of headline findings
top_category = category_summary["revenue"].idxmax()
top_region = region_summary["revenue"].idxmax()
slowest_region = delivery_summary["avg_delivery_days"].idxmax()
highest_return_category = return_by_category["return_rate"].idxmax()
negative_profit_share = (df["profit_margin"] < 0).mean()

print(f"1. Highest-revenue category: {top_category}")
print(f"2. Highest-revenue region: {top_region}")
print(f"3. Slowest region by average delivery time: {slowest_region}")
print(f"4. Highest return-rate category: {highest_return_category}")
print(f"5. Orders with negative profit metric: {negative_profit_share:.2%}")


### Business Analyst interpretation framework

Use the outputs above to turn the analysis into decisions:

- **Revenue growth:** prioritize high-revenue categories and regions while checking profitability.
- **Profitability:** investigate categories/products with high revenue but weak or negative profit.
- **Customer strategy:** identify high-value customers and age groups for targeted retention.
- **Operations:** investigate regions with longer delivery times.
- **Returns:** focus product categories with elevated return rates.
- **Discounting:** evaluate whether higher discounts increase sales enough to compensate for profitability pressure.
- **Management action:** convert these findings into owners, actions, KPIs and expected outcomes in the Business Analysis section.


## 16. Next Step

The outputs from this notebook should feed into:

1. **Power BI dashboards**
2. **Business Analysis findings**
3. **Profitability Matrix**
4. **Management Action Plan**
5. **Final project report**
6. **Project presentation**

This keeps the Python EDA, Power BI analysis and BA recommendations aligned around the same business questions.
